# Minimum-Volatility and Maximum-Sharpe Portfolios

Compare three static, long-only allocation rules from the portfolio experiments. The objectives are minimum volatility, minimum volatility with a minimum expected return, and maximum Sharpe ratio. All weights sum to one.

The example uses a configurable universe of public ETFs and downloads dividend-adjusted closes from Yahoo Finance. Allocation and evaluation use the same sample, so the curves illustrate the fitted portfolios. Daily portfolio returns assume constant weights and omit trading costs.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize

from _market_data import download_ohlcv


TICKERS = ["SPY", "QQQ", "VTI", "TLT", "GLD"]
BENCHMARK = "URTH"
START_DATE = "2020-01-01"
END_DATE = "2025-01-01"
ANNUAL_RETURN_TARGET = 0.12
ANNUAL_RISK_FREE_RATE = 0.02
SESSIONS_PER_YEAR = 252


def fit_allocations(returns):
    annual_mean = returns.mean().mul(SESSIONS_PER_YEAR).to_numpy()
    covariance = returns.cov().mul(SESSIONS_PER_YEAR).to_numpy()
    bounds = [(0.0, 1.0)] * len(returns.columns)
    fully_invested = {"type": "eq", "fun": lambda weights: weights.sum() - 1.0}
    return_floor = {
        "type": "ineq",
        "fun": lambda weights: weights @ annual_mean - ANNUAL_RETURN_TARGET,
    }

    def variance(weights):
        return weights @ covariance @ weights

    def negative_sharpe(weights):
        return -(weights @ annual_mean - ANNUAL_RISK_FREE_RATE) / np.sqrt(variance(weights))

    objectives = {
        "Minimum Volatility": (variance, [fully_invested]),
        "Minimum Volatility with a Return Target": (variance, [fully_invested, return_floor]),
        "Maximum Sharpe": (negative_sharpe, [fully_invested]),
    }
    initial_weights = np.full(len(returns.columns), 1.0 / len(returns.columns))
    weights = {}
    for label, (objective, constraints) in objectives.items():
        result = minimize(
            objective,
            initial_weights,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"ftol": 1e-10},
        )
        if not result.success:
            raise RuntimeError(f"{label} optimization failed. Check the sample and target. {result.message}")
        weights[label] = result.x
    return pd.DataFrame(weights, index=returns.columns)


prices = pd.concat(
    {
        ticker: download_ohlcv(ticker, START_DATE, END_DATE, auto_adjust=True)["Close"]
        for ticker in [*TICKERS, BENCHMARK]
    },
    axis=1,
).dropna()
returns = prices.pct_change().dropna()
weights = fit_allocations(returns[TICKERS])
portfolio_returns = returns[TICKERS].dot(weights)
portfolio_returns[BENCHMARK] = returns[BENCHMARK]
equity = (1 + portfolio_returns).cumprod()
drawdown = equity.div(equity.cummax()).sub(1)
print(weights.round(4))

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
equity.plot(ax=axes[0], title="Growth of One Unit")
drawdown.plot(ax=axes[1], title="Drawdown")
fig.tight_layout()
plt.show()
